<p style="text-align: center">
<img src="../../assets/images/dtlogo.png" alt="Duckietown" width="50%">
</p>

In [ ]:
%load_ext autoreload
%autoreload 2
from IPython import get_ipython
ipy = get_ipython()
if ipy and not hasattr(ipy, "highlighting_style"):
    ipy.highlighting_style = None
import logging
logging.getLogger("PIL").setLevel(logging.WARNING)

import numpy as np
import matplotlib.pyplot as plt
import cv2
from braitenberg_agent.agent import rgb_from_jpg


%matplotlib inline

# Image filtering

Now we want to use our image manipulation techniques to do some basic filtering of the images. We'll learn how in this activity, and at the end, you will update the file `preprocessing.py` so that we can send images with highlighted duckies to our Braitenberg agent.

In particular, let's say we want to avoid hitting duckies.

We then need to highlight our duckies in the image.

Let's start by loading a test image to work with as we did in the last activity.

In [ ]:
# experiment with different images!
fn = '../../assets/samples/big-duck/big-duck-08.jpg'
image = rgb_from_jpg(fn)
plt.imshow(image);

Let's now convert it to [HSV color space](https://en.wikipedia.org/wiki/HSL_and_HSV) so that it is easier to filter by colors.


In [ ]:
hsv = cv2.cvtColor(image, cv2.COLOR_RGB2HSV)

In OpenCV, the convention is the following:

* Hue is usually measured in degrees, from 0 to 360, as shown in [this scale](../../assets/HUE.png). However OpenCV uses a scale 0 to 179 - multiply by 2 to get the actual Hue in degree. (so to get a hue of 180 you have to tell 90 to OpenCV).
* S and V are from 0 to 255.

Now let's apply a simple image processing technique to highlight the region corresponding to a certain color.


To do this, we start by defining a region in HSV space with the following lower/upper bounds:


In [ ]:
lower_hsv = np.array([171, 140, 0])
upper_hsv = np.array([179, 200, 255])

The two arrays have 3 components: H(ue), S(aturation), V(value).

The first array is the lower bounds; the second array is the upper bounds.

This means that the bounds we defined above will allow us to select the pixels that have:

* Hue between OpenCV value 171 and 179, which are regular Hue values 342 to 358.
* Saturation between 140 and 200.
* Value between 0 and 255.



The *hue* is closest to what we intuitively call "color" and the one to use to do simple processing.

This is how to use our upper and lower bound to do the filtering of the image and show the result:

In [ ]:
# Returns a matrix of 0 or 1 that satisfy the constraint (pixels within our defined bounds)
mask = cv2.inRange(hsv, lower_hsv, upper_hsv)

# Get a "masked image"
masked = cv2.bitwise_and(image, image, mask=mask)
plt.imshow(masked);

The bounds we just used highlight the beak. But we want the yellow of the duckie!

## Activity

Your task now is to change those numbers above so that we highlight the duckies.


We've created a tool for you to make this easier and suggest you use the following steps:

1. Use [this online color picker](https://pinetools.com/image-color-picker) to click around on the sample images in the `/assets/samples` directory and get familiar with what color corresponds to which HSV values. (You can use any other tool, the link is just for reference). This will set you up to narrow in on the duckie yellow more quickly. **Regardless of the color picker you use, always double-check whether it uses a different convention for HSV ranges. If it does, make sure to convert the result to OpenCV's convention**.

2. Run the cell below. It is the same filtering code as above, with sliders for the lower and upper
   H, S and V bounds: move them until only the duckies are left in the masked image. The current bounds
   are printed under the sliders, in the same format used by `preprocessing.py`. Use the drop-down menu
   to try the other sample images in the `assets/samples` directory. (The sliders are
   [ipywidgets](https://ipywidgets.readthedocs.io/); if you only see text instead of sliders, re-run the
   cell once the notebook has fully loaded.)

In [ ]:
import glob
import os
import ipywidgets as widgets


def apply_hsv_bounds(image: np.ndarray, lower_hsv, upper_hsv) -> np.ndarray:
    """Keep only the pixels of the RGB `image` whose HSV values lie within the bounds."""
    hsv = cv2.cvtColor(image, cv2.COLOR_RGB2HSV)
    mask = cv2.inRange(hsv, np.array(lower_hsv), np.array(upper_hsv))
    return cv2.bitwise_and(image, image, mask=mask)


def hsv_explorer(image_files, lower_hsv=(171, 140, 0), upper_hsv=(179, 200, 255)):
    """Sliders for the lower/upper HSV bounds, applied live to one of the `image_files`."""
    hsv_max = (179, 255, 255)
    low = [widgets.IntSlider(value=v, min=0, max=m, description=f"low {c}") for c, v, m in zip("HSV", lower_hsv, hsv_max)]
    high = [widgets.IntSlider(value=v, min=0, max=m, description=f"high {c}") for c, v, m in zip("HSV", upper_hsv, hsv_max)]
    picker = widgets.Dropdown(
        options=[(os.path.relpath(f, "../../assets/samples"), f) for f in image_files],
        value=image_files[0],
        description="image",
    )
    bounds = widgets.HTML()
    view = widgets.Image(format="jpeg", width=640)
    images = {}

    def render(_change=None):
        if picker.value not in images:
            images[picker.value] = rgb_from_jpg(picker.value)
        lo = [s.value for s in low]
        hi = [s.value for s in high]
        masked = apply_hsv_bounds(images[picker.value], lo, hi)
        _, jpeg = cv2.imencode(".jpg", cv2.cvtColor(masked, cv2.COLOR_RGB2BGR))
        view.value = jpeg.tobytes()
        bounds.value = f"<pre>lower_hsv = np.array({lo})\nupper_hsv = np.array({hi})</pre>"

    for w in low + high + [picker]:
        w.observe(render, names="value")
    render()
    return widgets.VBox([picker, widgets.HBox([widgets.VBox(low), widgets.VBox(high)]), bounds, view])


default_image = "../../assets/samples/big-duck/big-duck-15.jpg"
sample_images = [default_image] + sorted(f for f in glob.glob("../../assets/samples/*/*.jpg") if f != default_image)
hsv_explorer(sample_images)

3. Once you have a satisfactory result, open the file [preprocessing.py](../../packages/solution/preprocessing.py), and change the values of `lower_hsv` and `upper_hsv` to the ones that you found to highlight the duckie. This will be used by your Braitenberg agent in the next notebook.

For reference, this is what your end result should look like.

In [ ]:
result_img = '../../assets/samples/result.jpg'
result = rgb_from_jpg(result_img)
plt.imshow(result);

Now go on to the [third notebook](../03-Braitenberg/braitenberg03.ipynb).
